# Détection de sophismes par taxonomie

**Navigation** : [README](./README.md) | [2-formal >>](./Argumentation-05-Formal-Verification-Python.ipynb)

| Champ | Valeur |
|-------|--------|
| **Module** | SymbolicAI / Argument_Analysis |
| **Rung** | 1-informal (Epic #2137 Phase 1) |
| **Niveau** | Introduction |
| **Duree estimee** | 30 minutes |
| **Kernel** | Python 3 (pandas, semantic_kernel) |

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

- [ ] Charger la **taxonomie des sophismes** (CSV `argumentum_fallacies_taxonomy.csv`) et identifier les 7 familles de premier niveau
- [ ] Charger la **taxonomie soeur des vertus argumentatives** (CSV `argumentum_virtues_taxonomy.csv`, 223 cartes, schema distinct) et constater la **divergence de schema** entre les deux CSV (deliberation upstream Argumentum)
- [ ] Descendre d'un niveau dans l'arbre taxonomique (depth=2) pour une famille donnee, via un filtre pandas
- [ ] Construire un **detecteur déterministe** (sans LLM) base sur des mots-cles, et l'appliquer sur un texte synthetique neutre
- [ ] Distinguer le rôle du detecteur par mots-cles (ce rung) du rôle du LLM (section 5, puis rungs ulterieurs) pour l'extraction sémantique
- [ ] Brancher l'**entonnoir agentique** du tronc EPITA-IS (`FallacyWorkflowPlugin`) sur la **même** taxonomie, et lire sa **trace de navigation** : PK confirmee, chemin parcouru, **branches rejetees**
- [ ] Comparer les deux detecteurs sur un **même** texte synthetique : ce que le lexique trouve, rate ou invente face a l'agent

### Prerequis

- Python 3.10+ ; packages `pandas`, `semantic_kernel`, `python-dotenv` installes
- Pour la section 5 (entonnoir agentique) : une cle API LLM dans le `.env` de la serie -- voir la cellule d'ouverture de la section 5
- Bases de lecture d'un CSV avec pandas (`read_csv`, filtrage boolean)
- Aucune cle API requise pour les sections 1 a 4 (100% déterministes, zero appel LLM)
- **Une cle API est requise pour la section 5** (`OPENAI_API_KEY` + `OPENAI_CHAT_MODEL_ID`, voir `.env.example`) : c'est la premiere section du rung qui appelle reellement un LLM

> **Note anti-theatre** : ce rung charge **reellement** le CSV et execute un **vrai** filtrage pandas. Aucune sortie n'est simulee. Le detecteur par mots-cles est volontairement simpliste : son but est pedagogique (montrer la couche de matching), pas d'atteindre une haute precision. La section 5 fait un **vrai appel LLM** sur la **même** taxonomie et affiche la trace reelle de la navigation -- pas une trace reconstruite.

> **Confidentialite** : tous les exemples sont **synthetiques et neutres** (meteorologie, oiseaux, Socrate, phrases abstraites). Aucun corpus EPITA, aucun nom d'etudiant, aucun texte sensible. Le depot est public sur GitHub.

## 1. Introduction : la taxonomie des sophismes

Un **sophisme** (fallacy) est un argument qui semble valide mais qui ne l'est pas, soit parce que les premisses ne soutiennent pas la conclusion, soit parce qu'il manipule l'auditoire. Pour detecter automatiquement un sophisme, il faut d'abord une **taxonomie** : un arbre hierarchise qui classe les sophismes par familles, sous-familles, etc.

Ce notebook utilise la taxonomie du projet *Argumentum* (1408 entrees, post-resync verbatim upstream ac33f607 -- voir `data/NOTICE` et `data/NOTICE-VIRTUES`). Une **taxonomie soeur** des vertus argumentatives (`argumentum_virtues_taxonomy.csv`, 223 cartes) vit dans le même dossier et partage la même structure d'arbre depth 0..7. Les deux CSV sont organises comme des arbres de profondeur variable :

| `depth` | Rôle | Exemple |
|---------|------|--------|
| `0` | Meta-noeud racine | "Argument fallacieux" (1 seule ligne, a exclure) |
| `1` | **7 familles de premier niveau** | Insuffisance, Influence, Obstruction, ... |
| `2` | Sous-familles | Obstruction -> {Refus du debat, Sabotage, Ad hominem} |
| `3`+ | Sophismes spécifiques | Ad hominem -> circonstanciel, abusif, ... |

**Stance anti-theatre** : on charge le CSV avec `pandas.read_csv` et on filtre **reellement** les lignes. Le detecteur de la section 3 tourne sur de vraies listes de mots-cles appliquees a un texte synthetique — jamais de résultat en dur.

| Étape | Outil | Ce qu'il fait |
|-------|-------|---------------|
| Texte naturel -> sophisme identifie | Detecteur mots-cles (ce rung) | Matching chaîne, déterministe, peu précis |
| Texte naturel -> structure informelle | LLM (rung 3) | Extraction sémantique probabiliste |
| Structure -> verification logique | Tweety (rung 2) | Preuve sound & complete |

## 2. Charger la taxonomie et identifier les 7 familles

La cellule ci-dessous charge le CSV avec `pandas.read_csv(..., dtype=str)` — tout est traite comme chaîne pour eviter les coercitions numériques sur les colonnes `depth`, `path`, etc. On filtre ensuite `depth == '1'` pour obtenir les familles de premier niveau.

> **Pourquoi exclure `depth == '0'` ?** Il existe un meta-noeud racine "Argument fallacieux" (1 ligne) qui represente le concept general de sophisme. Ce n'est pas une famille operationnelle : on l'evicte pour ne garder que les 7 vraies familles.

In [1]:
# Cellule [2] - Charger la taxonomie via la shim argumentation_lib
import pandas as pd

# Le package argumentation_lib vit a cote du notebook ; il est importable
# directement (le dossier du notebook est sur sys.path au runtime). La shim
# resout le dossier data/ via __file__ (cwd-independant) -- on evite ainsi le
# chemin relatif 'data/...' qui leverait FileNotFoundError hors de ce dossier.
# Aucun guard de path dans la cellule : la robustesse cwd est portee par la
# shim (_paths.py, __file__-relative), comme au rung 2-formal (#3857).
from argumentation_lib import get_data_dir
CSV_PATH = get_data_dir() / "argumentum_fallacies_taxonomy.csv"
print(f"Taxonomie resolue via la shim : {CSV_PATH.name}")

df = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8-sig")  # BOM UTF-8 stripped (canonical upstream ac33f607)

# La racine depth=0 est un meta-noeud (1 ligne) : on l'exclut.
meta_root = df[df['depth'] == '0']
print(f"Total lignes CSV      : {len(df)}")
print(f"Meta-noeud depth=0    : {len(meta_root)} ligne(s) -> exclue(s) de la liste des familles")

# Les 7 familles de premier niveau
familles = df[df['depth'] == '1'][['Famille', 'desc_fr']].reset_index(drop=True)
print(f"Familles depth=1      : {len(familles)}")
print()
print('=== Les 7 familles de sophismes (depth=1) ===')
for i, row in familles.iterrows():
    num = i + 1
    desc = row['desc_fr'] if pd.notna(row['desc_fr']) else '(pas de description)'
    print(f"  {num}. {row['Famille']:<25} -- {desc}")


Taxonomie resolue via la shim : argumentum_fallacies_taxonomy.csv
Total lignes CSV      : 1408
Meta-noeud depth=0    : 1 ligne(s) -> exclue(s) de la liste des familles
Familles depth=1      : 7

=== Les 7 familles de sophismes (depth=1) ===
  1. Insuffisance              -- Les arguments que vous utilisez ne suffisent pas à démontrer votre propos.
  2. Influence                 -- Vous manipulez votre auditoire pour le persuader au lieu de le convaincre.
  3. Erreur mathématique       -- Votre raisonnement comporte des inexactitudes quantitatives.
  4. Erreur de raisonnement    -- Votre thèse repose sur un raisonnement incohérent.
  5. Abus de langage           -- Vous usez de subtilités linguistiques pour convaincre.
  6. Tricherie                 -- Vous vous affranchissez des règles tacites qui régissent un débat rationnel.
  7. Obstruction               -- Par un artifice, oratoire ou autre, vous faites en sorte que la discussion ne se déroule pas comme prévu.


### Interpretation : les 7 familles de la taxonomie

**Sortie obtenue** : 7 familles de depth=1, chacune avec une description courte.

| Famille | Idee générale |
|---------|---------------|
| **Insuffisance** | Les premisses ne suffisent pas a prouver la conclusion |
| **Influence** | Manipulation emotionnelle de l'auditoire au lieu d'argumenter |
| **Erreur mathematique** | Inexactitude quantitative (statistiques, proportions) |
| **Erreur de raisonnement** | Raisonnement incoherent logiquement |
| **Abus de langage** | Jeu sur les mots, ambiguite linguistique |
| **Tricherie** | Contournement des règles tacites du debat rationnel |
| **Obstruction** | La discussion est empechee de se derouler normalement |

> **Point cle** : le meta-noeud `depth=0` ("Argument fallacieux", 1 ligne) n'est PAS une famille. C'est la racine conceptuelle de tout l'arbre. Les 7 familles operationnelles commencent a `depth=1`.

In [2]:
# Cellule [4-bis] - Charger la taxonomie soeur des VERTUS argumentatives
#
# Meme repertoire, meme upstream ac33f607, schema distinct delibere.
# On utilise la MEME shim cwd-robuste ; on dispatche sur le filename.
# Note pedagogique : les colonnes "snake_case" (family_fr) sont incompatibles
# avec le "PascalCase" du CSV Fallacies (Famille). Tout consumer qui
# presume un schema unifie tombera sur un KeyError -- d'ou l'importance du
# dispatch filename-based documente en data/NOTICE-VIRTUES.
VERT_PATH = get_data_dir() / "argumentum_virtues_taxonomy.csv"
df_vert = pd.read_csv(VERT_PATH, dtype=str, encoding="utf-8-sig")  # BOM UTF-8 stripped

# Distribution des profondeurs (Vertues 0..7, vs Fallacies 0..10)
depth_dist_vert = df_vert['depth'].value_counts().sort_index()
print(f"Taxonomie Vertus resolue via la shim : {VERT_PATH.name}")
print(f"Total cartes Vertus  : {len(df_vert)} (Fallacies: {len(df)} -- facteur ~6.3x)")
print(f"Distribution des profondeurs (Vertues 0..7) :")
for d, n in depth_dist_vert.items():
    print(f"  depth={d}  -> {n} cartes")
print()
print("=== Premiers vertus (depth=1, familles de premier niveau) ===")
vert_familles = df_vert[df_vert['depth'] == '1'][['family_fr', 'description_fr']].reset_index(drop=True)
for i, row in vert_familles.iterrows():
    num = i + 1
    desc = row['description_fr'] if pd.notna(row['description_fr']) else '(pas de description)'
    print(f"  {num}. {row['family_fr']:<25} -- {desc}")


Taxonomie Vertus resolue via la shim : argumentum_virtues_taxonomy.csv
Total cartes Vertus  : 223 (Fallacies: 1408 -- facteur ~6.3x)
Distribution des profondeurs (Vertues 0..7) :
  depth=0  -> 1 cartes
  depth=1  -> 7 cartes
  depth=2  -> 21 cartes
  depth=3  -> 63 cartes
  depth=4  -> 69 cartes
  depth=5  -> 28 cartes
  depth=6  -> 15 cartes
  depth=7  -> 19 cartes

=== Premiers vertus (depth=1, familles de premier niveau) ===
  1. Argument pertinent        -- Argument adapté au contexte, appuyé par des preuves utiles à la conclusion.
  2. Présentation intègre      -- Principe d’une argumentation transparente et honnête, sans sophismes ni manipulations.
  3. Sens quantitatif          -- Employer les mathématiques avec précision pour structurer et analyser des arguments ou des données.
  4. Inférence maîtrisée       -- Méthode rigoureuse qui articule des prémisses acceptables et des règles d’inférence valides pour aboutir à une conclusion cohérente et justifiée.
  5. Justesse lexicale 

### Interpretation : divergence de schema entre Fallacies et Vertues

**Sortie obtenue** : 223 cartes Vertus, structure depth=0..7, avec une distribution pedagogiquement interessante (1 racine / 7 familles / 21 sous-familles / 63 ss-sous-familles, puis 69/28/15/19 aux niveaux 4..7). Cette distribution suit la **même progression que les Fallacies** pour les 3 premiers niveaux (1/7/21/63) mais diverge au-dela -- les Vertues sont un arbre **moins profond et moins dense**.

**Schema divergence** (deliberation upstream Argumentum -- preserve verbatim) :

| Aspect | Fallacies (cell [3]) | Vertues (cell [4-bis]) |
|--------|---------------------|------------------------|
| Lignes totales | 1408 (PK 0..1407) | 223 (pk 0..222) |
| Profondeur max | 10 (cas rares) | 7 |
| **Header naming** | **PascalCase mixte** (`PK`, `Famille`, `Famille_camelCase`, `Sous-Famille`) | **snake_case unifie** (`pk`, `family_fr`, `subfamily_fr`, `subsubfamily_fr`) |
| **Card body fields** | **multi-field** (`text_fr`, `LTfr`, `Lfr115`, `example_fr`, `Lxfr145` -- 6+ par langue) | **3-field simplifie** (`title_fr`, `description_fr`, `remark_fr`) |
| Multilingue | FR/EN/RU/PT/AR/ES/ZH/FA (mêmes 8 langues) | idem |
| Cross-link | `crossLink_PredatesOn` / `Denounces` / ... (8 cols) | idem |
| AIF | `AIF_skosDirectRef` / `ExceptionRef` / ... (4 cols) | idem |

**Implication consumer** : le `family_fr` des Vertues N'EXISTE PAS comme `Famille` dans les Fallacies (la cle est en snake_case, pas PascalCase). Le `text_fr` multi-field des Fallacies N'EXISTE PAS comme `title_fr`/`description_fr` dans les Vertues. **Tout code qui presume un schema unifie tombera sur un KeyError**. Les loaders doivent dispatcher sur le **filename** (comme ici : Fallacies via `argumentum_fallacies_taxonomy.csv`, Vertues via `argumentum_virtues_taxonomy.csv`), pas assumer une liste de colonnes commune. Cette divergence est documentee verbatim dans `data/NOTICE-VIRTUES` pour eviter la surprise silencieuse cote consumer.

> **Pourquoi `encoding="utf-8-sig"` ici aussi ?** Même raison que pour les Fallacies (cell [3]) : les deux CSV sont consignes verbatim upstream ac33f607 avec leur BOM UTF-8 d'origine (bytes `0xEF 0xBB 0xBF` en debut de fichier). Pandas 2.x detecte ce BOM automatiquement avec `encoding='utf-8'`, mais specifier `encoding='utf-8-sig'` est **explicite et documente** (per ai-01 design-gate msg-20260703T110001-uqono0, 2026-07-03) et permet une portabilite vers d'autres readers Python (`csv.reader`, `open().readlines()`).


## 3. Descente d'un niveau (depth=2) : la famille Obstruction

Pour comprendre comment l'arbre se ramifie, on choisit une famille et on regarde ses **sous-familles directes** (depth=2). Prenons **Obstruction** : la famille qui regroupe les sophismes ou l'on empeche le debat d'avoir lieu normalement.

On filtre avec deux conditions combinees par `&` : `Famille == 'Obstruction'` ET `depth == '2'`. Le résultat est un sous-DataFrame qu'on affiche avec ses colonnes utiles.

In [3]:
# Cellule [4] - Descente d'un niveau : sous-familles depth=2 de la famille Obstruction
# On filtre le DataFrame principal : famille choisie ET depth=2.
FAMILLE_DEMO = 'Obstruction'
sous_familles = df[(df['Famille'] == FAMILLE_DEMO) & (df['depth'] == '2')][
    ['path', 'Famille', 'Sous-Famille', 'text_fr', 'example_fr']
].reset_index(drop=True)

print(f"=== Sous-familles depth=2 de '{FAMILLE_DEMO}' ({len(sous_familles)} trouvees) ===")
for i, row in sous_familles.iterrows():
    text_fr = row['text_fr'] if pd.notna(row['text_fr']) else '(sans nom)'
    example = row['example_fr'] if pd.notna(row['example_fr']) else '(pas d exemple)'
    ex_str = str(example)
    print(f"\n  [{row['path']}] {text_fr}")
    print(f"      Exemple : {ex_str[:90]}{'...' if len(ex_str) > 90 else ''}")

=== Sous-familles depth=2 de 'Obstruction' (3 trouvees) ===

  [7.1] Refus du débat
      Exemple : Mes positions sont claires, connues de tous, et je n’ai pas à en débattre.

  [7.2] Sabotage du débat
      Exemple : Allons plutôt dans mon bureau discuter de l’absurdité de votre demande d’augmentation…

  [7.3] Ad hominem
      Exemple : Comment pouvez-vous prétendre que votre politique de santé fonctionnera quand vous n’avez ...


### Interpretation : la ramification depth=2

**Sortie obtenue** : 3 sous-familles pour Obstruction, identifiees par leur `path` hiérarchique.

| path | Sous-famille | Idee |
|------|--------------|------|
| `7.1` | Refus du debat | Refuser d'argumenter (positions non negociables) |
| `7.2` | Saboter le debat (Sabotage) | Detourner la discussion, la rendre impossible |
| `7.3` | Ad hominem | Attaquer la personne plutot que l'argument |

**Points cles** :
1. Le `path` (ex `7.1`, `7.2`) encode la position hiérarchique : le `7` correspond a la 7e famille de depth=1 (Obstruction), le suffixe `.N` a la Neme sous-famille.
2. Chaque sous-famille peut elle-même se ramifier en depth=3 (ex : Ad hominem -> circonstanciel, abusif, preemptif). L'arbre a jusqu'a 10 niveaux pour certains sophismes très spécifiques.
3. La colonne `example_fr` donne un exemple d'usage — utile pour calibrer un detecteur ou entrainer un classifieur.

### Exercice 1 : descendre d'un niveau pour une AUTRE famille

**Contexte** : on vient de descendre Obstruction. Le même filtre pandas fonctionne pour n'importe quelle famille.

**Objectif** : choisissez une famille parmi {Insuffisance, Influence, Erreur mathematique, Erreur de raisonnement, Abus de langage, Tricherie} et affichez ses sous-familles depth=2 (nom + exemple). Retournez le nombre de sous-familles trouvees.

**Indices** :
- **Étape 1** : fixer `FAMILLE_EX1` a un nom parmi les 6 autres familles
- **Étape 2** : reutiliser le filtre `df[(df['Famille'] == FAMILLE_EX1) & (df['depth'] == '2')]`
- **Étape 3** : iterer et afficher chaque sous-famille avec son `text_fr`
- **Étape 4** : retourner/compter le nombre de sous-familles

In [4]:
# Exercice 1 : descendre d'un niveau pour une autre famille que Obstruction.
# Etape 1 : choisir une famille (parmi Insuffisance, Influence, Erreur mathematique,
#            Erreur de raisonnement, Abus de langage, Tricherie).
FAMILLE_EX1 = None  # TODO etudiant : remplacer par le nom d'une famille

# Etape 2-3 : filtrer et afficher les sous-familles depth=2 de cette famille.
# TODO etudiant : reprendre le filtre de la cellule demo avec FAMILLE_EX1.

# Etape 4 : nombre de sous-familles trouvees
nb_sous_familles_ex1 = None  # TODO etudiant : len(...) du DataFrame filtre

print(f"Exercice 1 : {FAMILLE_EX1} a {nb_sous_familles_ex1} sous-famille(s) depth=2")
print("Exercice a completer")

Exercice 1 : None a None sous-famille(s) depth=2
Exercice a completer


## 4. Detection déterministe par mots-cles sur un texte synthetique

La descente taxonomique nous donne un **referentiel** de noms de sophismes. On construit maintenant un **detecteur** : etant donne un texte, quel sophisme de la taxonomie est probablement present ?

Ce detecteur est **déterministe et zero-LLM** : la table `{nom_sophisme: mots_cles}` est construite depuis le CSV lui-meme (3 tokens discriminants de `desc_fr` par noeud exploitable), et un mot present suffit a signaler le sophisme. Cette couche est volontairement simpliste : elle ignore la **structure** de l'arbre, et c'est exactement ce qui motivera la section 5, ou un agent **navigue** dans cette meme taxonomie.

> **Ce detecteur est le comparateur**, pas la reponse. La section 5 le confronte au meme texte que l'entonnoir agentique : ce qu'il trouve, ce qu'il rate, ce qu'il invente.

> **Important** : on travaille sur des **exemples synthetiques** (phrases abstraites construites pour la demonstration). Aucun texte reel, aucune donnee sensible.


In [5]:
# Cellule [6] - Detecteur deterministe par mots-cles + test sur un texte synthetique
#
# Audit #18391 : la version precedente declarait un dico a 4 entrees en dur, sans
# aucun ancrage dans la taxonomie de 1408 noeuds chargee par c.3. On construit
# maintenant la table depuis `df` : pour chaque noeud qui porte un
# `nom_vulgarise` non vide, on prend 3 tokens discriminants de `desc_fr`
# (longueur >= 5, alphabetique, lowercase, hors stopwords). 38 sophismes
# couverts au lieu de 4 -- c'est la portee reelle de la taxonomie.

import re

_STOPWORDS = {
    "vous", "votre", "cette", "celui", "celle", "celles", "votre",
    "leurs", "leurs", "toute", "tous", "etre", "ayant", "sont", "sommes",
    "faire", "soit", "meme", "alors", "dans", "pour", "sans", "sous",
    "avec", "mais", "donc", "plus", "moins", "entre", "comme", "aussi",
    "leur", "bien", "peut", "etre",
}


def _build_detecteur(taxonomy_df, max_keywords=3, min_len=5):
    """Construit la table {nom_vulgarise: (Famille, Sous-Famille, [keywords])}.

    Les keywords sont des tokens de `desc_fr` (lowercase, alpha, longueur
    >= ``min_len``, hors stopwords) pris dans l'ordre d'apparition (jusqu'a `max_keywords`), sans tri -- la selection des tokens discriminants reste une borne du detecteur (voir Limite 2 de la cellule d'interpretation).
    """
    table = {}
    for _, row in taxonomy_df.iterrows():
        nom = row.get("nom_vulgarisé")
        # Filtrer les entrees polluees du CSV upstream (PK 993 porte un
        # commentaire CSV dans la colonne nom_vulgarise).
        if not isinstance(nom, str) or not nom.strip() or len(nom.strip()) > 50:
            continue
        desc = row.get("desc_fr") or ""
        tokens = [
            tok
            for tok in re.findall(r"[a-zA-ZaeiouyAEIOUY]{%d,}" % min_len, desc.lower())
            if tok not in _STOPWORDS
        ]
        # Dedup en gardant l'ordre
        seen = set()
        kws = []
        for tok in tokens:
            if tok in seen:
                continue
            seen.add(tok)
            kws.append(tok)
            if len(kws) >= max_keywords:
                break
        if not kws:
            continue
        famille = row.get("Famille") or ""
        sous_famille = row.get("Sous-Famille") or ""
        table[nom.strip()] = (famille, sous_famille, kws)
    return table


DETECTEUR_SOPHISMES = _build_detecteur(df)
print(f"Detecteur : {len(DETECTEUR_SOPHISMES)} sophismes couverts depuis la taxonomie.")
print(f"Exemples : {list(DETECTEUR_SOPHISMES.keys())[:5]}")


def detecter_sophismes(texte, table=DETECTEUR_SOPHISMES):
    """Retourne la liste des sophismes detectes dans `texte`.

    Chaque element est un dict {sophisme, famille, sous_famille, mot_cle_trigger}.
    Aucun appel LLM : c'est du filtrage de chaine pur sur les ``keywords``
    produits par ``_build_detecteur``.
    """
    texte_lower = texte.lower()
    matches = []
    for nom_sophisme, (famille, sous_famille, mots_cles) in table.items():
        for mot in mots_cles:
            if mot in texte_lower:
                matches.append({
                    "sophisme": nom_sophisme,
                    "famille": famille,
                    "sous_famille": sous_famille,
                    "mot_cle_trigger": mot,
                })
                break  # un seul trigger par sophisme suffit
    return matches


# Texte synthetique neutre : pas de personne reelle, pas d opinion politique.
TEXTE_SYNTHETIQUE = (
    "Vous etes incompetent pour juger, et de toute facon je rejette "
    "toute information qui pourrait contredire ma position."
)

detections = detecter_sophismes(TEXTE_SYNTHETIQUE)
print("--- Texte synthetique analyse ---")
print(TEXTE_SYNTHETIQUE)
print()
print(f"--- Sophismes detectes ({len(detections)}) ---")
for d in detections[:10]:
    print(f"  - {d['sophisme']:<35} (famille={d['famille']}, sous_famille={d['sous_famille']}) "
          f"trigger='{d['mot_cle_trigger']}'")
if len(detections) > 10:
    print(f"  ... et {len(detections) - 10} autre(s)")


Detecteur : 39 sophismes couverts depuis la taxonomie.


Exemples : ['La tête dans le sable', 'Répulsif anti-girafe', 'Trouver des excuses', 'Inertie mentale', 'Inventer des salades']
--- Texte synthetique analyse ---
Vous etes incompetent pour juger, et de toute facon je rejette toute information qui pourrait contredire ma position.

--- Sophismes detectes (4) ---
  - La tête dans le sable               (famille=Insuffisance, sous_famille=Argument bâclé) trigger='information'
  - Versatilité                         (famille=Tricherie, sous_famille=Changement de cap) trigger='position'
  - Rétropédalage                       (famille=Tricherie, sous_famille=Changement de cap) trigger='position'
  - Biais culturel                      (famille=Tricherie, sous_famille=Raisonnement biaisé) trigger='position'


### Interpretation : sortie du detecteur

**Sortie obtenue** : 4 sophismes signales sur le texte synthetique, chacun avec son trigger issu du matching mots-cles sur la table construite depuis la taxonomie.

| Sophisme | Famille / Sous-Famille | Trigger | Lecture |
|----------|------------------------|---------|--------|
| La tete dans le sable | Insuffisance / Argument bâclé | `information` | Refuser l'information qui contredit sa position est une posture d'evitement |
| Versatilité | Tricherie / Changement de cap | `position` | Invoquer le changement de position de l'adversaire pour disqualifier son propos |
| Rétropédalage | Tricherie / Changement de cap | `position` | Rejeter en bloc ce qui menace sa position sans examiner le fond |
| Biais culturel | Tricherie / Raisonnement biaisé | `position` | Rejeter un argument au nom d'une appartenance culturelle presupposee |

**Pourquoi 4 maintenant, et pas 3 : diagnostic de derive.** La table est reconstruite dynamiquement depuis le CSV (`_build_detecteur(df)`) -- son contenu suit le snapshot de taxonomie. La re-synchronisation upstream du CSV (re-pin AIF, #19086) a **nettoye le champ `nom_vulgarisé` de la ligne Versatilité**, qui portait un commentaire editorial en clair (56 caracteres) -- le filtre anti-pollution du constructeur (`nom.strip()` de plus de 50 caracteres), documente dans le code, l'excluait donc de la table. Nettoyee, la ligne entre dans la table (38 -> 39 sophismes couverts) et son mot-cle `position` tire sur le texte synthetique : c'est la quatrieme detection. Meme cause pour le libelle : la sous-famille « Pensée biaisée » de Biais culturel a ete renommee « Raisonnement biaisé » en amont. La derive vient du **CSV courant**, pas du code -- les sorties ci-dessus sont celles de la re-execution sur ce snapshot, et le compte bougera encore si la taxonomie evolue.

**Limites volontaires de ce detecteur** :
1. **Pas de contexte** : le matching mots-cles ignore la semantique ; `information` peut designer un fait neutre, pas forcement un refus. Un meme trigger (`position`) alimente trois lignes distinctes : la table ne distingue pas les sens.
2. **Selection des tokens figee** : les mots-cles sont les premiers tokens discriminants de `desc_fr` (longueur >= 5, hors stopwords), pris dans l'ordre -- pas les plus discriminants statistiquement.
3. **Texte synthetique unique** : le detecteur n'est eprouve que sur une phrase construite pour contenir les mots-cles -- aucune pretention de generalisation.

### Exercice 2 : ajouter une règle de detection pour un sophisme au choix

**Contexte** : la table `DETECTEUR_SOPHISMES` est construite dynamiquement a partir de la taxonomie (39 entrees a ce snapshot, derivees des noeuds dont `nom_vulgarise` est exploitable -- le compte suit le CSV courant : voir le diagnostic de derive de la section 4). Pour ajouter un sophisme absent de la table, la voie directe est d'enrichir la taxonomie source : le generateur reconstruira la table au prochain appel de `_build_detecteur(df)`.

**Objectif** : ajoutez une nouvelle entree directement dans le dictionnaire `DETECTEUR_SOPHISMES` (un sophisme de votre choix parmi ceux de la taxonomie), definissez 2-3 mots-cles pertinents, puis testez votre detecteur etendu sur une phrase synthetique que vous ecrivez. La forme du tuple `(famille, sous_famille, [keywords])` reste inchangee.

**Exemples de sophismes candidats** (a chercher dans le CSV filtre par `nom_vulgarisé`) : Pente glissante, Homme de paille, Appel a l'emotion, Argument circulaire.

**Indices** :
- **Étape 1** : copier `DETECTEUR_SOPHISMES` dans une nouvelle variable `table_etendue`
- **Étape 2** : ajouter une cle avec un tuple `(famille, sous_famille, [mots_cles])`
- **Étape 3** : ecrire une `PHRASE_TEST` synthetique contenant un des mots-cles
- **Étape 4** : appeler `detecter_sophismes(PHRASE_TEST, table_etendue)` et afficher

In [6]:
# Exercice 2 : etendre le detecteur et tester sur une phrase synthetique.
# Etape 1 : copier la table existante
table_etendue = dict(DETECTEUR_SOPHISMES)  # TODO etudiant : point de depart

# Etape 2 : ajouter un nouveau sophisme
# TODO etudiant : table_etendue['Mon sophisme'] = ('Famille', 'Sous-famille', ['mot1', 'mot2'])

# Etape 3 : phrase synthetique de test
PHRASE_TEST = None  # TODO etudiant : une phrase contenant un des nouveaux mots-cles

# Etape 4 : detection et affichage
detections_ex2 = []  # TODO etudiant : detecter_sophismes(PHRASE_TEST, table_etendue)
print(f"Exercice 2 : {len(detections_ex2)} sophisme(s) detecte(s)")
print("Exercice a completer")

Exercice 2 : 0 sophisme(s) detecte(s)
Exercice a completer


### Exercice 3 : fonction qui retourne la famille du premier sophisme detecte

**Contexte** : on veut parfois une reponse simple — "quelle est la famille dominante de ce texte ?" — plutot qu'une liste de tous les matches.

**Objectif** : ecrivez une fonction `famille_premier_sophisme(texte)` qui retourne le nom de la **famille** du premier sophisme detecte dans `texte`, ou `None` si aucun match. Utilisez `detecter_sophismes` définie plus haut.

**Indices** :
- **Étape 1** : appeler `detecter_sophismes(texte)` pour obtenir la liste des matches
- **Étape 2** : si la liste est vide, retourner `None`
- **Étape 3** : sinon, retourner `matches[0]['famille']`
- **Étape 4** : tester sur deux phrases (une qui match, une qui ne match pas)

In [7]:
# Exercice 3 : famille du premier sophisme detecte.
def famille_premier_sophisme(texte):
    # TODO etudiant : utiliser detecter_sophismes(texte) et retourner
    # la famille du premier element, ou None si la liste est vide.
    # Etape 1 :
    matches = []  # TODO etudiant : detecter_sophismes(texte)
    # Etape 2-3 :
    return None  # TODO etudiant : remplacer par la logique

# Etape 4 : tests
phrase_avec = "Tous les chats sont noirs, c'est bien connu."
phrase_sans = "Le ciel est bleu aujourd'hui."
famille_avec = famille_premier_sophisme(phrase_avec)
famille_sans = famille_premier_sophisme(phrase_sans)
print(f"Phrase avec sophisme -> famille : {famille_avec}")
print(f"Phrase sans sophisme  -> famille : {famille_sans}")
print("Exercice a completer")

Phrase avec sophisme -> famille : None
Phrase sans sophisme  -> famille : None
Exercice a completer


## 5. Detection agentique : l'entonnoir taxonomique

La section 4 a produit une **baseline** : une table `{nom_sophisme: mots_cles}` construite depuis la taxonomie, mais qui ignore la **structure** de l'arbre. Un mot present suffit, et le sophisme est declare -- sans jamais regarder *ou* le noeud se situe, ni *pourquoi*.

Le tronc EPITA-IS traite le meme probleme autrement, et c'est ce que cette section branche : un **entonnoir agentique** qui fait *naviguer* un LLM dans la taxonomie. Le composant est vendore dans `argumentation_lib/_fallacy_workflow_plugin.py` (`FallacyWorkflowPlugin` ; provenance et ecarts declares dans `argumentation_lib/NOTICE-EPITA`, section « Entonnoir de sophismes »). Il lit **le meme CSV** que la section 2.

| Phase | Ce qui se passe | Ce qui la borne |
|-------|-----------------|-----------------|
| **1. Filet large** | Le LLM liste *tous* les sophismes qu'il croit reconnaitre, sans contrainte. Chaque nom est resolu en **PK** de la taxonomie, via un index des noeuds de profondeur 1 a 3. | Si **aucun** nom ne se resout, le plugin **retombe en silence sur le regime `one_shot`** -- le LLM repond alors sans naviguer, et la trace est vide (voir la cellule [5.2], qui le signale) |
| **2. Approfondissement** | **Tous** les candidats resous en phase 1 sont lances (jusqu'a `MAX_CANDIDATES = 20` ; le chemin `wide_net_parallel` ne tranche pas par `MAX_BRANCHES`). Pour chaque candidat, un agent **esclave** dont les **seuls outils** sont `explore_branch`, `confirm_fallacy` et `conclude_no_fallacy`. Il ne peut pas repondre librement : a chaque noeud il **confirme** ou **descend**, jusqu'a une feuille. | Circuit breaker **par branche** : `MAX_NAVIGATION_LLM_CALLS = 18` appels ; budget **global** de la descente : `DESCENT_TOTAL_CALL_BUDGET = 240` appels pour l'analyse entiere (env `FALLACY_DESCENT_CALL_BUDGET`) ; fan-out borne (`SUBBRANCH_FANOUT_WIDTH = 2` par fourche, `SUBBRANCH_FANOUT_BUDGET = 12` par analyse) ; confirmation a profondeur >= 2 |
| **3. Trace** | Le resultat porte le **chemin de navigation** : PK confirmee, chaine de noeuds traverses, et les **branches explorees puis rejetees**, avec leur motif. | — |

C'est la **trace** qui fait la valeur pedagogique : elle montre *comment* l'agent se deplace dans un referentiel de 1408 noeuds, et pas seulement *ce qu'il repond*. Un agent qui se contente de repondre ne produit **aucun rejet** ; un agent qui navigue en produit beaucoup.

> **Cette section est la premiere du rung a appeler reellement un LLM.** Elle exige une cle API (voir `Prerequis`). Sans cle, la cellule **echoue avec un message explicite** : elle ne se rabat **jamais** en silence sur le lexique de la section 4.


In [8]:
# Cellule [5.1] - Entonnoir agentique (FallacyWorkflowPlugin) sur le MEME texte
#
# Le plugin est vendorise depuis le tronc EPITA-IS (voir NOTICE-EPITA, section
# "Entonnoir de sophismes"). Il lit le MEME CSV que la section 2, via la meme
# shim -- aucun chemin en dur.
import json
import logging
import os

from dotenv import load_dotenv

# Le .env vit a cote de data/ : on le localise via la shim (cwd-independant),
# comme la cellule [2] le fait pour le CSV.
AA_DIR = get_data_dir().parent
load_dotenv(AA_DIR / ".env", override=True)

# Piege mesure : une variable OPENAI_BASE_URL PRESENTE mais VIDE fait construire
# au connecteur Semantic Kernel une URL vide (APIConnectionError). On la retire
# dans ce cas precis, pour laisser le defaut du service s'appliquer.
if not (os.getenv("OPENAI_BASE_URL") or "").strip():
    os.environ.pop("OPENAI_BASE_URL", None)

# --- Garde de cle : echec EXPLICITE, jamais de repli silencieux sur le lexique.
_api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
if not _api_key:
    raise RuntimeError(
        "La section 5 (entonnoir agentique) exige OPENAI_API_KEY. "
        "Aucun repli silencieux n'est effectue : la section 4 (lexique "
        "mots-cles) reste la baseline utilisable sans cle API."
    )

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

import argumentation_lib._epita_vendor_shim  # noqa: F401  (effet de bord sys.modules)
from argumentation_analysis.plugins.fallacy_workflow_plugin import FallacyWorkflowPlugin

MODELE_LLM = os.getenv("OPENAI_CHAT_MODEL_ID")
kernel_llm = Kernel()
kernel_llm.add_service(OpenAIChatCompletion(
    service_id="global_llm_service", ai_model_id=MODELE_LLM, api_key=_api_key,
))

# Journal de descente : le plugin loggue chaque decision de l'agent (exploration,
# rejet, confirmation). On l'ecoute pour pouvoir afficher la trace a la cellule
# suivante -- c'est la matiere pedagogique de cette section.
journal_descente = []


class _CollecteurDescente(logging.Handler):
    """Collecte les messages du logger du plugin dans `journal_descente`."""

    def emit(self, record):
        journal_descente.append(record.getMessage())


plugin_entonnoir = FallacyWorkflowPlugin(
    master_kernel=kernel_llm,
    llm_service=kernel_llm.get_service("global_llm_service"),
    taxonomy_file_path=str(CSV_PATH),  # le MEME CSV que la section 2
)
plugin_entonnoir.logger.addHandler(_CollecteurDescente())
# Sans niveau explicite, les messages INFO du plugin n'atteignent PAS le
# collecteur : un logger sans niveau propre herite du root (WARNING par
# defaut), et la trace afficherait alors "0 rejet" alors que l'agent a bel et
# bien navigue. Un silence se lirait comme une preuve d'absence de navigation.
plugin_entonnoir.logger.setLevel(logging.INFO)

print(f"Modele LLM        : {MODELE_LLM}")
print(f"Taxonomie         : {CSV_PATH.name}")
print(f"Index de profondeur : {len(plugin_entonnoir._build_deep_index(3))} noeuds (depth 1-3)")
print()
print("Analyse guidee en cours (compter 1 a 3 minutes)...")
reponse_entonnoir = await plugin_entonnoir.run_guided_analysis(  # noqa: F704
    argument_text=TEXTE_SYNTHETIQUE
)
resultat_entonnoir = json.loads(reponse_entonnoir)
print("Termine.")


Modele LLM        : gpt-5-mini
Taxonomie         : argumentum_fallacies_taxonomy.csv
Index de profondeur : 91 noeuds (depth 1-3)

Analyse guidee en cours (compter 1 a 3 minutes)...


Termine.


In [9]:
# Cellule [5.2] - Lire la trace : regime, cout, chemin, branches REJETEES
#
# L'acceptation de l'issue #18391 tient en une phrase : "la trace imprimee
# contient au moins une branche exploree puis rejetee -- preuve que l'agent a
# navigue, et n'a pas seulement repondu".
METH_RUN = resultat_entonnoir.get("exploration_method")
REGIME_RUN = resultat_entonnoir.get("analysis_regime")
NB_BRANCHES = resultat_entonnoir.get("branches_explored")
NB_ITER = resultat_entonnoir.get("total_iterations")
NB_APPELS = resultat_entonnoir.get("descent_calls_made")

print("=== Regime d'exploration ===")
print(f"  methode            : {METH_RUN}")
print(f"  regime             : {REGIME_RUN}")
print(f"  branches explorees : {NB_BRANCHES}")
print(f"  iterations         : {NB_ITER}")
print(f"  appels LLM         : {NB_APPELS}")

if REGIME_RUN != "funnel":
    print()
    print("  [ATTENTION] Le regime n'est PAS 'funnel' : le plugin a bascule sur son repli")
    print("      silencieux 'one_shot'. Le LLM a REPONDU sans naviguer, et la trace")
    print("      ci-dessous est vide. C'est le symptome d'un index de taxonomie vide")
    print("      (aucun nom candidat resolu en PK) -- la degradation est muette par")
    print("      construction, d'ou cet avertissement explicite.")

# --- Les trois familles d'evenements de la descente -------------------------
_abandonnees = [m.strip() for m in journal_descente if m.strip().startswith("Branch abandoned")]
_non_confirmees = [m.strip() for m in journal_descente if m.strip().startswith("Leaf not confirmed")]
_supersessions = [m.strip() for m in journal_descente if "supersession" in m]

print()
print("=== Descente : ce que l'agent a explore PUIS rejete ===")
print(f"  branches abandonnees   : {len(_abandonnees)}")
print(f"  feuilles non confirmees: {len(_non_confirmees)}")
for _m in (_abandonnees + _non_confirmees)[:6]:
    print(f"    - {_m[:165]}")
_restants = len(_abandonnees) + len(_non_confirmees) - 6
if _restants > 0:
    print(f"    ... et {_restants} autre(s) rejet(s)")
for _m in _supersessions:
    print(f"  {_m}")

# --- Les conclusions retenues, avec leur POSITION reelle dans l'arbre ------
taxo = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8-sig").set_index("PK")
print()
print("=== Conclusions confirmees (position reelle dans la taxonomie) ===")
print(f"  {'PK':>6}  {'depth':>5}  {'path':<14} nom")
pks_entonnoir = []
for _f in resultat_entonnoir.get("fallacies", []):
    _pk = str(_f.get("taxonomy_pk"))
    pks_entonnoir.append(_pk)
    if _pk in taxo.index:
        _r = taxo.loc[_pk]
        print(f"  {_pk:>6}  {_r['depth']:>5}  {str(_r['path']):<14} {_r['text_fr']}")
    else:
        print(f"  {_pk:>6}  {'?':>5}  {'?':<14} (PK absente du CSV !)")

_profondeurs = [int(taxo.loc[p]["depth"]) for p in pks_entonnoir if p in taxo.index]
if _profondeurs:
    print()
    print(f"  Profondeurs des PK confirmees : {sorted(_profondeurs)}")
    print(f"  Toutes a profondeur >= 2 (critere d'acceptation) : "
          f"{all(d >= 2 for d in _profondeurs)}")


=== Regime d'exploration ===
  methode            : wide_net_parallel
  regime             : funnel
  branches explorees : 6
  iterations         : 25
  appels LLM         : 51

=== Descente : ce que l'agent a explore PUIS rejete ===
  branches abandonnees   : 4
  feuilles non confirmees: 4
    - Branch abandoned: Le texte n’est pas une répétition excessive visant à imposer un point de vue. Il attaque la compétence de l’interlocuteur (“Vous êtes incompetent”)
    - Branch abandoned: Le texte n'exprime pas une valorisation excessive de la science. Il attaque la compétence de l'interlocuteur et refuse systématiquement toute infor
    - Branch abandoned: The statement attacks the opponent's competence and explicitly rejects any contrary information, which is an abusive ad hominem/closed-mindedness, 
    - Branch abandoned: Le passage attaque la compétence de l’interlocuteur («Vous êtes incompétent pour juger») et refuse toute information contradictoire («je rejette to
    - Leaf not confi

In [10]:
# Cellule [5.3] - Les deux detecteurs sur le MEME texte : ce que le lexique
# trouve, rate, ou invente
#
# Comparaison sur les PK, pas sur les libelles : le lexique indexe ses entrees
# par `nom_vulgarise`, l'entonnoir par le `text_fr` de la PK confirmee -- deux
# champs differents du meme CSV. Comparer les libelles serait un faux desaccord.
nom_vers_pk = {
    str(r["nom_vulgarisé"]).strip(): str(r["PK"])
    for _, r in df.iterrows()
    if isinstance(r.get("nom_vulgarisé"), str) and str(r["nom_vulgarisé"]).strip()
}
pks_lexique = sorted({nom_vers_pk[d["sophisme"]] for d in detections if d["sophisme"] in nom_vers_pk})
pks_entonnoir = sorted({str(f.get("taxonomy_pk")) for f in resultat_entonnoir.get("fallacies", [])})

_communs = sorted(set(pks_lexique) & set(pks_entonnoir))
_lexique_seul = sorted(set(pks_lexique) - set(pks_entonnoir))
_entonnoir_seul = sorted(set(pks_entonnoir) - set(pks_lexique))


def _libelle(pk):
    if pk not in taxo.index:
        return f"PK {pk} (absente du CSV)"
    r = taxo.loc[pk]
    return f"PK {pk:>4} (depth={r['depth']}, {str(r['path']):<11}) {r['text_fr']}"


print("=== Comparaison sur le meme texte synthetique ===")
print(f"  lexique mots-cles (section 4) : {len(pks_lexique):>2} sophisme(s)")
print(f"  entonnoir agentique (section 5): {len(pks_entonnoir):>2} sophisme(s)")
print(f"  accord (meme PK)               : {len(_communs):>2}")
print()
print(f"--- Confirmes par l'entonnoir, RATES par le lexique ({len(_entonnoir_seul)}) ---")
for _pk in _entonnoir_seul:
    print(f"  + {_libelle(_pk)}")
print()
print(f"--- Signales par le lexique, NON confirmes par l'entonnoir ({len(_lexique_seul)}) ---")
for _pk in _lexique_seul:
    print(f"  - {_libelle(_pk)}")
print()
if _communs:
    print(f"--- Les deux s'accordent ({len(_communs)}) ---")
    for _pk in _communs:
        print(f"  = {_libelle(_pk)}")
else:
    print("--- Aucun accord : les deux detecteurs ne designent AUCUNE meme PK. ---")


=== Comparaison sur le meme texte synthetique ===
  lexique mots-cles (section 4) :  4 sophisme(s)
  entonnoir agentique (section 5):  9 sophisme(s)
  accord (meme PK)               :  0

--- Confirmes par l'entonnoir, RATES par le lexique (9) ---
  + PK 1307 (depth=6, 7.1.3.5.1.1) Dénialisme
  + PK 1352 (depth=3, 7.2.3      ) Empoisonnement du puits
  + PK 1384 (depth=5, 7.3.2.2.2  ) Appel au biais
  + PK 1399 (depth=4, 7.3.3.1    ) Argument tonal
  + PK  358 (depth=4, 2.3.1.1    ) Cadrage
  + PK  431 (depth=4, 2.3.2.2    ) Prise de contrôle
  + PK  465 (depth=6, 2.3.2.3.1.3) Démoralisation
  + PK  770 (depth=4, 4.3.1.3    ) Acte de foi
  + PK  963 (depth=6, 6.1.3.1.2.1) Argument par la lecture sélective

--- Signales par le lexique, NON confirmes par l'entonnoir (4) ---
  - PK 1004 (depth=4, 6.2.2.5    ) Couverture
  - PK 1174 (depth=3, 6.3.2      ) Biais culturels
  - PK    5 (depth=5, 1.1.1.1.1  ) Ignorance délibérée
  - PK  992 (depth=3, 6.2.2      ) Vouloir le beurre et l’argent 

### Lecture de la trace : ce que « naviguer » veut dire

Trois choses se lisent dans la sortie precedente, et aucune n'est visible avec un detecteur par mots-cles.

1. **Un regime, pas un score.** `analysis_regime = "funnel"` dit que l'agent a **navigue**. Le repli `one_shot` (le LLM repond d'un coup, sans arbre) est le mode degrade du meme composant : si le regime affiche n'est pas `funnel`, la trace est vide et **rien** de ce qui suit ne vaut -- c'est pourquoi la cellule [5.2] le signale explicitement plutot que de l'afficher discretement.

2. **Des branches rejetees, avec leur motif.** Les lignes `Branch abandoned` et `Leaf not confirmed` sont la preuve de navigation : a chaque noeud, l'agent a examine une branche et l'a ecartee *en expliquant pourquoi* (« le texte n'invoque aucune autorite », « ce n'est pas une question insinuant une accusation »). Un agent qui se contente de repondre ne produit **jamais** ce journal. Ces motifs sont aussi, pedagogiquement, la partie la plus interessante : ils montrent le raisonnement taxonomique a l'oeuvre.

3. **Une position, pas un nom.** Chaque conclusion porte une **PK** et un **`path`** reels (`7.3.2.3.3`, `1.1.3.4`, ...), donc une **profondeur**. Le lexique de la section 4 rend un nom, sans position : c'est la difference structurelle entre les deux approches.

> **Le cout est reel et borne.** `descent_calls_made` compte les appels LLM reellement consommes par la descente ; le circuit breaker `MAX_NAVIGATION_LLM_CALLS = 18` plafonne **chaque branche**, et le budget global `DESCENT_TOTAL_CALL_BUDGET = 240` plafonne **l'analyse entiere**. La trace ci-dessus (6 branches explorees, 51 appels) est donc dans les clous des deux etages -- environ 8,5 appels par branche sous le breaker individuel, 51 sous le budget global : aucun depassement. Une analyse guidee n'est pas gratuite : c'est le prix de la navigation, et il se compare au cout nul (mais a la precision faible) du lexique.

> **Les resultats varient d'une execution a l'autre.** L'entonnoir fait de vrais appels LLM : deux executions du meme texte peuvent confirmer des PK differentes, et le nombre de rejets change. C'est une propriete du composant, pas un defaut du notebook -- et c'est precisement pourquoi la trace est affichee *telle qu'elle a ete produite*, plutot que remplacee par un resultat attendu. Le lexique de la section 4, lui, est parfaitement reproductible : la comparaison oppose donc une methode deterministe a une methode stochastique.

> **Sur la precision de l'entonnoir.** Les motifs de rejet sont souvent plus convaincants que les conclusions retenues : en ecartant une branche, l'agent explique parfois quel aurait ete le bon label. Un agent qui navigue rend son raisonnement **inspectable**, ce qui permet de le contester -- il ne le rend pas juste pour autant. La comparaison de la cellule [5.3] se lit donc comme un **diagnostic**, jamais comme un verdict de superiorite d'une methode sur l'autre.

> **Un mot sur la comparaison.** Les deux detecteurs partagent la **meme taxonomie** et le **meme texte**. La cellule [5.3] compare leurs sorties **sur les PK** -- et non sur les libelles, qui different par construction (le lexique indexe par `nom_vulgarise`, l'entonnoir rend le `text_fr` de la PK). Un accord nul, s'il est mesure, n'est pas un bug : c'est le resultat honnete de deux methodes qui **partagent la meme entree** (le meme texte, la meme taxonomie) mais **different par leur mecanisme** (matching lexical deterministe contre navigation agentique) **et leur critere de decision** (presence d'un mot-cle contre confirmation explicite a une feuille de l'arbre).


## 6. Pont vers l'etat partage (shim argumentation_lib)

Jusqu'ici, les detections des sections 4 et 5 vivent dans des **listes Python locales**. Dans le pipeline complet de la serie, les analyses (informelle, formelle, orchestree) partagent un **etat unique** -- l'objet `RhetoricalAnalysisState` fourni par la shim `argumentation_lib`. C'est l'objet que le rung [3-orchestration](./Argumentation-07-Orchestration-Python.ipynb) remplit via un agent LLM, et que le rung [2-formal](./Argumentation-05-Formal-Verification-Python.ipynb) enrichit de résultats de preuve.

Cette section montre le **geste fondamental** du pipeline : nourrir cet etat partage a partir des **deux** detecteurs du rung -- le lexique déterministe (section 4) et l'entonnoir agentique (section 5). On retrouve ainsi, cote a cote dans la **même** structure de données, deux methodes qui partagent la même entrée (le texte synthetique, la taxonomie CSV) mais different par leur mecanisme et leur critere de décision.

| Champ de l'etat | Rempli ici | Rempli au rung 3 (LLM) |
|-----------------|------------|------------------------|
| `raw_text` | le texte synthetique | un texte reel fourni en entree |
| `identified_arguments` | un argument de demonstration | les arguments extraits semantiquement |
| `identified_fallacies` | les detections du **lexique** (section 4) **et** les conclusions de l'**entonnoir** (section 5), chacune avec sa famille et son `path` reel | les sophismes extraits par le LLM |

> **Pourquoi le `path` reel, et pas le nom de la sous-famille ?** Le rung 2 (logique formelle) et le rung 3 raisonnent sur la **position** d'un sophisme dans l'arbre, pas sur son libelle. Le `path` (ex. `7.3.2.3.3`) est la seule cle qui survive a un renommage de `text_fr` -- c'est lui qui rend l'etat partage exploitable en aval.


In [11]:
# Cellule [section 6] - Pont vers l'etat partage : les DEUX detecteurs
#
# Le lexique (section 4) et l'entonnoir (section 5) produisent chacun une liste
# Python locale. Dans le pipeline complet de la serie, les analyses partagent un
# ETAT UNIQUE -- l'objet RhetoricalAnalysisState de la shim argumentation_lib.
# Le rung 3 le remplit via un agent LLM ; le rung 2 y attache des resultats de
# preuve. Ici on montre le geste fondamental : y verser les deux detecteurs de
# ce rung, sans orchestrateur.
import logging

from argumentation_lib import RhetoricalAnalysisState

# Rendre les logs internes de l'etat silencieux (niveau WARNING) pour un output propre.
logging.getLogger("RhetoricalAnalysisState").setLevel(logging.WARNING)


def _path_reel(pk, repli=""):
    """Le `path` reel de la taxonomie pour une PK -- cle stable de l'arbre.

    On ne verse JAMAIS un nom de sous-famille dans `taxonomy_path` : le rung 2 et
    le rung 3 raisonnent sur la position dans l'arbre, et un libelle ne survit
    pas a un renommage de `text_fr`.
    """
    return str(taxo.loc[pk]["path"]) if pk in taxo.index else repli


etat = RhetoricalAnalysisState(initial_text=TEXTE_SYNTHETIQUE)
arg_id = etat.add_argument("Argument synthetique analyse par les deux detecteurs du rung")

# --- Volet 1 : le lexique mots-cles (section 4) ---
for d in detections:
    etat.add_fallacy(
        fallacy_type=d["sophisme"],
        justification=f"lexique mots-cles, declencheur '{d['mot_cle_trigger']}'",
        target_arg_id=arg_id,
        family=d["famille"],
        taxonomy_path=_path_reel(nom_vers_pk.get(d["sophisme"], ""), d["sous_famille"]),
    )

# --- Volet 2 : l'entonnoir agentique (section 5) ---
for f in resultat_entonnoir.get("fallacies", []):
    pk = str(f.get("taxonomy_pk"))
    ligne = taxo.loc[pk] if pk in taxo.index else None
    etat.add_fallacy(
        fallacy_type=f.get("fallacy_type"),
        justification=f"entonnoir agentique, PK {pk} confirmee a profondeur "
                      f"{ligne['depth'] if ligne is not None else '?'}",
        target_arg_id=arg_id,
        family=(ligne["Famille"] if ligne is not None else None),
        taxonomy_path=_path_reel(pk),
    )

snapshot = etat.get_state_snapshot(summarize=True)
print("--- Etat partage apres les DEUX detecteurs ---")
print(f"  raw_text             : {snapshot['raw_text_snippet']}")
print(f"  arguments identifies : {snapshot['argument_count']}")
print(f"  sophismes identifies : {snapshot['fallacy_count']}")
print()
print(f"  {'source':<22} {'type':<34} {'path':<13} cible")
for fid, fdata in etat.identified_fallacies.items():
    just = fdata.get("justification", "")
    src = "lexique" if just.startswith("lexique") else "entonnoir"
    print(f"  {src:<22} {str(fdata['type'])[:33]:<34} "
          f"{str(fdata.get('taxonomy_path', '?')):<13} {fdata.get('target_argument_id', '?')}")
print()
print("Cet etat est exactement la structure que le rung 3 (orchestration LLM)")
print("remplira via un agent, et que le rung 2 (formel) enrichira de resultats de preuve.")


--- Etat partage apres les DEUX detecteurs ---
  raw_text             : Vous etes incompetent pour juger, et de toute facon je rejette toute information qui pourrait contredire ma position.
  arguments identifies : 1
  sophismes identifies : 13

  source                 type                               path          cible
  lexique                La tête dans le sable              1.1.1.1.1     arg_1
  lexique                Versatilité                        6.2.2         arg_1
  lexique                Rétropédalage                      6.2.2.5       arg_1
  lexique                Biais culturel                     6.3.2         arg_1
  entonnoir              Appel au biais                     7.3.2.2.2     arg_1
  entonnoir              Argument tonal                     7.3.3.1       arg_1
  entonnoir              Prise de contrôle                  2.3.2.2       arg_1
  entonnoir              Argument par la lecture sélective  6.1.3.1.2.1   arg_1
  entonnoir              Démoralis

## 7. Conclusion et recapitulatif

Ce rung a pose les **fondations taxonomiques** de la detection de sophismes, puis a confronte **deux methodes** sur la **meme** taxonomie et le **meme** texte : d'un cote un **lexique mots-cles** construit depuis le CSV (sections 1 a 4, 100% déterministe), de l'autre l'**entonnoir agentique** du tronc EPITA-IS (section 5), qui fait *naviguer* un LLM dans l'arbre et rend sa **trace de navigation**. La section 6 a verse les deux dans l'**etat partage** de la serie (`RhetoricalAnalysisState`). Aucune sortie n'est simulee : le lexique est un vrai filtrage pandas, l'entonnoir un vrai appel LLM dont la trace (branches rejetees, chemin, profondeur) est affichee telle qu'elle a ete produite.

### Tableau recapitulatif

| Étape | Outil | Force | Limite |
|-------|-------|-------|-------|
| **Charger la taxonomie** | `pandas.read_csv` + `argumentation_lib.get_data_dir()` | Referentiel de 1408 sophismes + 223 vertus verifiees, path cwd-robuste | Le CSV est vaste, il faut filtrer |
| **Charger la taxonomie soeur Vertues** | Même shim, dispatch filename-based | Schema distinct delibere (snake_case, 3-field body, 223 cartes) | Pas d'unified schema inter-CSV -- KeyError si presume |
| **Descendre d'un niveau** | Filtre booléen `(Famille == X) & (depth == '2')` | Identification des sous-familles directes | Manuelle, famille par famille |
| **Detection mots-cles** | Table `{sophisme: [mots]}` construite depuis `desc_fr` + `in` | Déterministe, zero API, transparent, cout nul | Pas de contexte, faible precision, **aucune notion de position** dans l'arbre |
| **Entonnoir agentique** | `FallacyWorkflowPlugin.run_guided_analysis` (SK + LLM) | **Navigue** dans la taxonomie ; rend une PK, un `path`, une profondeur et les **branches rejetees** avec leur motif | Cout reel (appels LLM bornes par `MAX_BRANCHES` / `MAX_NAVIGATION_LLM_CALLS`), exige une cle API, et **retombe en silence sur `one_shot`** si aucun candidat ne se resout en PK |
| **Etats partage** | `RhetoricalAnalysisState.add_fallacy(...)` | Verse les **deux** detecteurs dans la meme structure, avec le `path` reel | La structure reste vide tant qu'aucun agent LLM ne l'enrichit |

### Points a retenir

1. **La taxonomie est un arbre** : `depth` encode le niveau (0=racine meta, 1=familles, 2=sous-familles, 3+=instances spécifiques). Le `path` (ex `7.3`) encode la position hiérarchique.

2. **Le detecteur mots-cles est une baseline mesuree** : il montre ce qu'on peut faire sans LLM, et donc ce que l'agent apporte. Ses limites sont l'absence de contexte (`tous les oiseaux` declencherait une fausse "Generalisation abusive") et l'**absence de position** : il rend un nom, jamais un `path`.

3. **La trace est le livrable de l'entonnoir** : un agent qui navigue produit des **branches rejetees** avec leur motif ; un agent qui se contente de repondre n'en produit aucune. C'est la difference observable entre `analysis_regime = "funnel"` et le repli `one_shot` -- que la section 5 signale explicitement plutot que de l'afficher discretement.

4. **Confidentialite par design** : tous les exemples sont synthetiques. Pour un usage reel, le pipeline complet (rung 3) orchestre l'extraction LLM puis la verification formelle (rung 2) -- jamais de stockage de texte sensible dans le depot public.

5. **L'etat partage est le lien entre les rungs** : `RhetoricalAnalysisState` est le conteneur commun. Ce rung (detection), le rung 2 (preuve) et le rung 3 (orchestration LLM) y lisent et ecrivent. Brancher notre baseline des maintenant rend cette architecture concrete.

### Prochaines étapes

- [2-formal](./Argumentation-05-Formal-Verification-Python.ipynb) : une fois un argument identifie, verifier sa **validite logique** avec le raisonneur Tweety (PL, FOL, modal, argumentation de Dung).
- [3-orchestration](./Argumentation-07-Orchestration-Python.ipynb) : remplacer la table mots-cles par une orchestration multi-agents, puis deleguer la verification a Tweety.